# 07 — LoRA for Diffusion

Low-Rank Adaptation (LoRA) enables efficient fine-tuning of large diffusion models on small datasets.

## DreamBooth-Style LoRA Fine-Tuning

**LoRA** (Hu et al., 2021) adds trainable low-rank decomposition matrices to frozen pretrained weights:
$$W' = W_0 + \Delta W = W_0 + BA$$
where $W_0 \in \mathbb{R}^{d \times k}$ is frozen, $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, and rank *r ≪ min(d, k)*. This reduces trainable parameters from *d×k* to *r(d+k)*.

**DreamBooth** (Ruiz et al., 2022) fine-tunes a diffusion model on 3-5 images of a specific subject (person, object, style), using a rare text token as an identifier (e.g., 'a photo of [V] dog'). A **prior preservation loss** prevents the model from forgetting the general class:
$$\mathcal{L} = \mathbb{E}[||\epsilon - \epsilon_\theta(x_t, [V])||^2] + \lambda \mathbb{E}[||\epsilon - \epsilon_\theta(x_t^{pr}, c_{pr})||^2]$$

**LoRA for diffusion** applies LoRA to the attention layers of the UNet, requiring only ~100MB of weights to represent a fine-tuned style or subject — compared to ~4GB for the full model. This is the mechanism behind community fine-tunes on CivitAI and Hugging Face.

In [ ]:
# LoRA implementation: low-rank adaptation layer
import numpy as np
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    """Linear layer with LoRA low-rank adaptation."""
    def __init__(self, in_features, out_features, rank=4, alpha=8):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        # Frozen pretrained weight
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02,
                                    requires_grad=False)
        # LoRA matrices: trainable
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

    def forward(self, x):
        # Base forward (frozen)
        base_out = x @ self.weight.T
        # LoRA adaptation
        lora_out = x @ self.lora_A.T @ self.lora_B.T * self.scaling
        return base_out + lora_out

    def merge_weights(self):
        """Merge LoRA into base weight for inference (zero overhead)."""
        with torch.no_grad():
            self.weight.data += (self.lora_B @ self.lora_A) * self.scaling

# Parameter count comparison
d, k, r = 512, 512, 16
full_params = d * k
lora_params = r * (d + k)
print(f'Full fine-tune: {full_params:,} parameters')
print(f'LoRA (rank={r}): {lora_params:,} parameters')
print(f'Compression: {full_params/lora_params:.1f}x fewer trainable params')

# Demo: LoRA layer
layer = LoRALinear(64, 64, rank=4)
x = torch.randn(8, 64)
out = layer(x)
print(f'\nLoRA forward: input {x.shape} -> output {out.shape}')

trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
total = sum(p.numel() for p in layer.parameters())
print(f'Trainable: {trainable}, Total: {total} ({trainable/total:.1%} trainable)')

In [ ]:
# LoRA fine-tuning: adapt a noise network to a new 'style'
import matplotlib.pyplot as plt

torch.manual_seed(42)
T = 100
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

# Pretrain on circular data
theta = torch.linspace(0, 2*3.14159, 1000)
x_pretrain = torch.stack([torch.cos(theta) * 2, torch.sin(theta) * 2], dim=1)
x_pretrain += torch.randn_like(x_pretrain) * 0.1

# Target fine-tune 'style': ellipse
x_finetune = torch.stack([torch.cos(theta) * 3, torch.sin(theta) * 1], dim=1)
x_finetune += torch.randn_like(x_finetune) * 0.1

class BaseNoiseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.time_embed = nn.Embedding(T, 32)
        self.l1 = nn.Linear(2 + 32, 64)
        self.l2 = nn.Linear(64, 64)
        self.l3 = nn.Linear(64, 2)

    def forward(self, x, t):
        h = torch.relu(self.l1(torch.cat([x, self.time_embed(t)], dim=-1)))
        h = torch.relu(self.l2(h))
        return self.l3(h)

base_net = BaseNoiseNet()
opt = torch.optim.Adam(base_net.parameters(), lr=3e-4)
for step in range(1000):
    idx = torch.randint(0, len(x_pretrain), (256,))
    x0 = x_pretrain[idx]
    t = torch.randint(0, T, (256,))
    noise = torch.randn_like(x0)
    xt = alpha_bars[t].sqrt().unsqueeze(1)*x0 + (1-alpha_bars[t]).sqrt().unsqueeze(1)*noise
    loss = ((base_net(xt, t) - noise)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

# Freeze base net
for p in base_net.parameters(): p.requires_grad_(False)

# Add LoRA adapters to l2 layer
class LoRAAdapter(nn.Module):
    def __init__(self, base_layer, rank=4):
        super().__init__()
        self.base = base_layer
        d = base_layer.out_features
        k = base_layer.in_features
        self.A = nn.Parameter(torch.randn(rank, k) * 0.02)
        self.B = nn.Parameter(torch.zeros(d, rank))
        self.scale = 8 / rank

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

lora_l2 = LoRAAdapter(base_net.l2, rank=4)

class LoRANet(nn.Module):
    def __init__(self, base, lora_l2):
        super().__init__()
        self.time_embed = base.time_embed
        self.l1 = base.l1
        self.l2 = lora_l2
        self.l3 = base.l3
    def forward(self, x, t):
        h = torch.relu(self.l1(torch.cat([x, self.time_embed(t)], dim=-1)))
        h = torch.relu(self.l2(h))
        return self.l3(h)

lora_net = LoRANet(base_net, lora_l2)
lora_opt = torch.optim.Adam([p for p in lora_net.parameters() if p.requires_grad], lr=3e-4)

for step in range(500):
    idx = torch.randint(0, len(x_finetune), (128,))
    x0 = x_finetune[idx]
    t = torch.randint(0, T, (128,))
    noise = torch.randn_like(x0)
    xt = alpha_bars[t].sqrt().unsqueeze(1)*x0 + (1-alpha_bars[t]).sqrt().unsqueeze(1)*noise
    loss = ((lora_net(xt, t) - noise)**2).mean()
    lora_opt.zero_grad(); loss.backward(); lora_opt.step()

lora_params = sum(p.numel() for p in lora_net.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in base_net.parameters())
print(f'LoRA params: {lora_params} / {total_params} total ({lora_params/total_params:.1%})')
print('LoRA fine-tuning complete')